# Small-Sample LightGBM Baseline

In [ ]:
from pathlib import Path
import gc

import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEAN_PATH = PROJECT_ROOT / "data" / "clean" / "partition_8_drop_missing" / "part-0.parquet"
RAW_PATH = PROJECT_ROOT / "data" / "part_8.parquet"
PARQUET_PATH = CLEAN_PATH if CLEAN_PATH.exists() else RAW_PATH

TARGET_COL = "responder_6"
WEIGHT_COL = "weight"
DATE_COL = "date_id"
EXTRA_FEATURE_COLS = ["symbol_id", "time_id"]
CATEGORICAL_COLS = ["symbol_id", "feature_09", "feature_10", "feature_11"]

MAX_ROWS = 100_000
BATCH_SIZE = 16_384
SEED = 42

PARQUET_PATH

In [ ]:
dataset = ds.dataset(PARQUET_PATH, format="parquet")
columns = dataset.schema.names
feature_cols = [col for col in columns if col.startswith("feature_")]
extra_cols = [col for col in EXTRA_FEATURE_COLS if col in columns]
model_cols = feature_cols + extra_cols
categorical_cols = [col for col in CATEGORICAL_COLS if col in model_cols]

required_cols = [TARGET_COL, WEIGHT_COL, DATE_COL]
missing_cols = [col for col in required_cols if col not in columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

print(f"path: {PARQUET_PATH}")
print(f"features: {len(feature_cols)}")
print(f"model columns: {len(model_cols)}")
print(f"categorical columns: {categorical_cols}")

In [ ]:
def collect_sample(parquet_path, columns, max_rows=100_000, batch_size=16_384):
    dataset = ds.dataset(parquet_path, format="parquet")
    scanner = dataset.scanner(
        columns=columns,
        batch_size=batch_size,
        batch_readahead=1,
        fragment_readahead=1,
        use_threads=True,
    )

    frames = []
    rows = 0
    for batch in scanner.to_batches():
        df_batch = batch.to_pandas(split_blocks=True, self_destruct=True)
        df_batch = df_batch.dropna(subset=columns)
        if len(df_batch) == 0:
            continue
        remaining = max_rows - rows
        frames.append(df_batch.head(remaining))
        rows += len(frames[-1])
        if rows >= max_rows:
            break

    if not frames:
        raise ValueError("No rows collected after dropping missing values")
    return pd.concat(frames, ignore_index=True)


sample_cols = [DATE_COL] + model_cols + [WEIGHT_COL, TARGET_COL]
df = collect_sample(PARQUET_PATH, sample_cols, max_rows=MAX_ROWS, batch_size=BATCH_SIZE)
gc.collect()

print(df.shape)
print(df[[DATE_COL, WEIGHT_COL, TARGET_COL]].describe())

In [ ]:
for col in categorical_cols:
    df[col] = df[col].astype("category")

unique_dates = np.array(sorted(df[DATE_COL].unique()))
val_date_count = max(1, min(10, int(np.ceil(len(unique_dates) * 0.2))))
val_dates = unique_dates[-val_date_count:]

train_mask = df[DATE_COL] < val_dates[0]
val_mask = df[DATE_COL] >= val_dates[0]

print({
    "unique_dates": len(unique_dates),
    "train_rows": int(train_mask.sum()),
    "val_rows": int(val_mask.sum()),
    "train_date_min": int(df.loc[train_mask, DATE_COL].min()),
    "train_date_max": int(df.loc[train_mask, DATE_COL].max()),
    "val_date_min": int(df.loc[val_mask, DATE_COL].min()),
    "val_date_max": int(df.loc[val_mask, DATE_COL].max()),
})

In [ ]:
def weighted_r2(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weight = np.asarray(weight, dtype=np.float64)
    numerator = np.sum(weight * np.square(y_true - y_pred))
    denominator = np.sum(weight * np.square(y_true))
    return np.nan if denominator == 0 else 1.0 - numerator / denominator


train_set = lgb.Dataset(
    df.loc[train_mask, model_cols],
    label=df.loc[train_mask, TARGET_COL],
    weight=df.loc[train_mask, WEIGHT_COL],
    feature_name=model_cols,
    categorical_feature=[col for col in categorical_cols if col in model_cols],
    free_raw_data=False,
)

val_set = lgb.Dataset(
    df.loc[val_mask, model_cols],
    label=df.loc[val_mask, TARGET_COL],
    weight=df.loc[val_mask, WEIGHT_COL],
    feature_name=model_cols,
    categorical_feature=[col for col in categorical_cols if col in model_cols],
    free_raw_data=False,
)

In [ ]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "device_type": "cpu",
    "num_leaves": 64,
    "learning_rate": 0.03,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,
    "min_data_in_leaf": 200,
    "lambda_l2": 1.0,
    "seed": SEED,
    "feature_pre_filter": False,
    "verbosity": -1,
}

model = lgb.train(
    params,
    train_set,
    valid_sets=[train_set, val_set],
    valid_names=["train", "valid"],
    num_boost_round=500,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=25),
    ],
)

preds = model.predict(df.loc[val_mask, model_cols], num_iteration=model.best_iteration)
score = weighted_r2(df.loc[val_mask, TARGET_COL], preds, df.loc[val_mask, WEIGHT_COL])

print(f"best_iteration: {model.best_iteration}")
print(f"valid_weighted_r2: {score:.8f}")

In [ ]:
importance = (
    pd.DataFrame({
        "feature": model.feature_name(),
        "importance_gain": model.feature_importance(importance_type="gain"),
        "importance_split": model.feature_importance(importance_type="split"),
    })
    .sort_values(["importance_gain", "importance_split"], ascending=False)
    .reset_index(drop=True)
)

top20 = importance.head(20)["feature"].tolist()
print(top20)
importance.head(25)

In [ ]:
top20_cols = top20
top20_categorical_cols = [col for col in categorical_cols if col in top20_cols]
top20_df = df[[DATE_COL, WEIGHT_COL, TARGET_COL] + top20_cols].copy()

print(top20_df.shape)
print(top20_df.columns.tolist())
print(top20_df[top20_cols + [WEIGHT_COL, TARGET_COL]].isna().sum().sum())


In [ ]:
top20_train_set = lgb.Dataset(
    top20_df.loc[train_mask, top20_cols],
    label=top20_df.loc[train_mask, TARGET_COL],
    weight=top20_df.loc[train_mask, WEIGHT_COL],
    feature_name=top20_cols,
    categorical_feature=top20_categorical_cols,
    free_raw_data=False,
)

top20_val_set = lgb.Dataset(
    top20_df.loc[val_mask, top20_cols],
    label=top20_df.loc[val_mask, TARGET_COL],
    weight=top20_df.loc[val_mask, WEIGHT_COL],
    feature_name=top20_cols,
    categorical_feature=top20_categorical_cols,
    free_raw_data=False,
)

top20_model = lgb.train(
    params,
    top20_train_set,
    valid_sets=[top20_train_set, top20_val_set],
    valid_names=["train", "valid"],
    num_boost_round=500,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=25),
    ],
)

top20_preds = top20_model.predict(top20_df.loc[val_mask, top20_cols], num_iteration=top20_model.best_iteration)
top20_score = weighted_r2(top20_df.loc[val_mask, TARGET_COL], top20_preds, top20_df.loc[val_mask, WEIGHT_COL])

print(f"top20_best_iteration: {top20_model.best_iteration}")
print(f"top20_valid_weighted_r2: {top20_score:.8f}")
